In [37]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from pybamm import exp
from pybamm import tanh

Defining Model Variables

In [38]:
CL_atom = pybamm.Variable(
    "concentration of netutral lithium atoms in the SEI [mol.m-3]",  domain=["SEI layer"])
CL_ion = pybamm.Variable(
    "concentration of llithium ions in the SEI [mol.m-3]",  domain=["SEI layer"])
Phi_SEI = pybamm.Variable("Potential in the SEI [V]",  domain=["SEI layer"])
L_SEI = pybamm.Variable("Thickness of SEI [m]",  domain=["SEI layer"])


x = pybamm.SpatialVariable(
    "x", domain=["SEI layer"], coord_sys="cartesian")

Defining parameters of the model

In [39]:
T = pybamm.Parameter('Initial temperature [K]')
R = pybamm.Parameter('Ideal gas constant [J.K-1.mol-1]')
F = pybamm.Parameter("Faraday constant [C.mol-1]")
D_Li_atom = pybamm.Parameter(
    'diffusion coefficient of netutral lithium atoms in the SEI [m2.s-1]')
D_Li_ion = pybamm.Parameter(
    'diffusion coefficient of llithium ions in the SEI [m2.s-1]')

CL_atom_0 = pybamm.Parameter(
    "initial concentration of netutral lithium atoms in the SEI [mol.m-3]")
CL_ion_0 = pybamm.Parameter(
    "initial concentration of llithium ions in the SEI [mol.m-3]")

Phi_SEI_0 = pybamm.Parameter("initial potential in the SEI [V]")


M_SEI = pybamm.Parameter("molar weight of SEI material [kg.mol-1]")
rho_SEI = pybamm.Parameter("density of SEI material [kg.m-3]")
n_SEI = pybamm.Parameter("electrone number [-]")


L_tun = pybamm.Parameter("length of the tunneling [m]")
L_SEI_0 = pybamm.Parameter("initial thickness of SEI [m]")

CL_ion_max = pybamm.Parameter(
    "maximum concentration of llithium ions in the electrolyte [mol.m-3]")

In [40]:
param = pybamm.ParameterValues(
    {
        'Initial temperature [K]': 298.15,
        'Ideal gas constant [J.K-1.mol-1]': 8.314462618,
        "Faraday constant [C.mol-1]": 96485.33212,
        'diffusion coefficient of netutral lithium atoms in the SEI [m2.s-1]': 1,
        'diffusion coefficient of llithium ions in the SEI [m2.s-1]': 29866.0,
        'initial concentration of netutral lithium atoms in the SEI [mol.m-3]': 0,
        'initial concentration of llithium ions in the SEI [mol.m-3]': 1,
        'initial potential in the SEI [V]': 0,
        'molar weight of SEI material [kg.mol-1]': 0.162,
        'density of SEI material [kg.m-3]': 1690,
        'electrone number [-]': 2,
        'length of the tunneling [m]': 2e-9,
        'initial thickness of SEI [m]': 1e-9,
        'maximum concentration of llithium ions in the electrolyte [mol.m-3]': 1000

    }
)

In [41]:
NL_ions = D_Li_ion / L_SEI * pybamm.grad(CL_ion) - D_Li_ion * F/(
    R*T * L_SEI)*pybamm.grad(Phi_SEI)  # define the flux for lithoum ions
# define the flux for netutral lithium atoms
NL_atom = D_Li_atom / L_SEI * pybamm.grad(CL_atom)

V_SEI = M_SEI/(n_SEI*rho_SEI)

# Unclear how to define the source terms
A = 1
R_CLi_ions = 0
R_CLi_atom = 0


J_Li_0 = NL_atom - D_Li_atom*F/(R*T * L_SEI)*CL_atom*pybamm.grad(Phi_SEI)
J_tun = A * J_Li_0 * pybamm.exp(- L_SEI / L_tun)
Je = J_Li_0 + J_tun


dL_SEI_dt = - V_SEI / F * pybamm.inner(x, Je)
dCL_ion_dt = 1/L_SEI * dL_SEI_dt * \
    pybamm.inner(x, pybamm.grad(CL_ion)) - 1 / L_SEI * pybamm.div(NL_ions) + \
    R_CLi_ions  # define the rhs equation
dCL_atom_dt = 1/L_SEI * dL_SEI_dt * \
    pybamm.inner(x, pybamm.grad(CL_ion))-1/L_SEI * pybamm.div(NL_atom) + \
    R_CLi_atom  # define the rhs equation

In [42]:
model = pybamm.lithium_ion.BaseModel()

model.algebraic = {Phi_SEI: pybamm.div(Je)}
model.rhs = {CL_ion: dCL_ion_dt, CL_atom: dCL_atom_dt, L_SEI: dL_SEI_dt}

In [43]:
model.variables = {
    "concentration of netutral lithium atoms in the SEI [mol.m-3]": CL_atom,
    "concentration of llithium ions in the SEI [mol.m-3]": CL_ion,
    "Potential in the SEI [V]": Phi_SEI,
    "Thickness of SEI [m]": L_SEI
}

In [44]:
model.initial_conditions = {CL_ion: CL_ion_0, CL_atom: CL_atom_0,
                            Phi_SEI: Phi_SEI_0, L_SEI: L_SEI_0}

j_int = 2  # Prescribed
lbc_CL_ion = j_int/(F*D_Li_ion)
rbc_CL_ion = CL_ion_max
V_input = -1.1
model.boundary_conditions = {CL_ion: {"left": (lbc_CL_ion, "Neumann"), "right": (rbc_CL_ion, "Dirichlet")},
                             CL_atom: {"left": (0, "Neumann"), "right": (0, "Dirichlet")},
                             Phi_SEI: {"left": (0, "Neumann"), "right": (V_input, "Neumann")}}
# model.boundary_conditions = {c: {"left": (lbc, "Neumann"), "right": (rbc, "Neumann")}}

In [45]:

geometry = pybamm.Geometry({"SEI layer": {x: {"min": 0, "max": 1}}})

In [46]:
param.process_model(model)
param.process_geometry(geometry)
submesh_types = {"SEI layer": pybamm.Uniform1DSubMesh}
var_pts = {x: 50}
# # create a mesh of our geometry, using a uniform grid with 20 volumes
mesh = pybamm.Mesh(geometry, submesh_types, var_pts)
spatial_methods = {"SEI layer": pybamm.FiniteVolume()}
disc = pybamm.Discretisation(mesh, spatial_methods)
disc.process_model(model)

In [47]:
# solver = pybamm.ScipySolver()
# pybamm.CasadiSolver(mode="fast")
solver = pybamm.IDAKLUSolver()

In [48]:
sim = pybamm.Simulation(
    model,
    geometry=geometry,
    parameter_values=param,
    var_pts=var_pts,
    spatial_methods=spatial_methods,
    solver=solver,
)

sim.solve([0, 1])


[IDAS ERROR]  IDACalcIC
  The linear solver setup failed unrecoverably.


[IDAS ERROR]  IDASolve
  At t = 0, the linear solver setup failed unrecoverably.



SolverError: idaklu solver failed